# UrbanFlow — Machine Learning & Explainable AI Pipeline
## 02. DBSCAN Spatial Clustering & Random Forest with SHAP Explanations

This notebook trains and evaluates:
1. DBSCAN density-based spatial clustering to discover accident hotspots in Mumbai.
2. Random Forest Severity Classifier with SHAP TreeExplainer feature attributions.
3. Diurnal Traffic Congestion Regressor.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import shap
import joblib

In [ ]:
# Load datasets
df_accidents = pd.read_csv('../datasets/sample/mumbai_accidents_historical.csv')
coords = np.radians(df_accidents[['latitude', 'longitude']].values)

# 500 meter cluster radius
kms_per_radian = 6371.0088
eps_rad = 0.5 / kms_per_radian

db = DBSCAN(eps=eps_rad, min_samples=15, metric='haversine')
df_accidents['cluster_id'] = db.fit_predict(coords)
n_clusters = len(set(df_accidents['cluster_id'])) - (1 if -1 in df_accidents['cluster_id'] else 0)
print(f"Discovered {n_clusters} spatial clusters in Mumbai.")

In [ ]:
# Inspect top clusters by fatal collisions
hotspots = df_accidents[df_accidents['cluster_id'] != -1].groupby('cluster_id').agg({
    'incident_id': 'count',
    'fatalities': 'sum',
    'corridor': lambda x: x.mode()[0],
    'latitude': 'mean',
    'longitude': 'mean'
}).rename(columns={'incident_id': 'accident_count'}).sort_values('fatalities', ascending=False)
hotspots.head(10)